# Capítulo 4: Clasificación de Textos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/clase4-clasificacion-textos.ipynb)

## Objetivos de aprendizaje

- Comprender el problema de clasificación supervisada aplicado a texto.
- Implementar clasificadores con Naive Bayes, SVM y Redes Neuronales.
- Evaluar modelos con métricas adecuadas (accuracy, precision, recall, F1).
- Aplicar técnicas de validación cruzada.

## 4.1 Clasificación supervisada de texto

A diferencia del clustering, la **clasificación** es una tarea **supervisada**: contamos con datos etiquetados para entrenar un modelo que luego predice la categoría de nuevos documentos.

### Ejemplos

- **Análisis de sentimiento**: Positivo / Negativo / Neutro
- **Detección de spam**: Spam / No spam
- **Categorización de noticias**: Deportes / Política / Tecnología
- **Clasificación de tickets**: Urgente / Normal / Bajo

### Pipeline de clasificación

```
Texto → Preprocesamiento → Vectorización (TF-IDF) → Modelo → Predicción
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

# Datos de ejemplo: reseñas de productos con sentimiento
textos = [
    # Positivas
    "Excelente producto, superó mis expectativas",
    "Muy buena calidad, lo recomiendo totalmente",
    "Increíble relación calidad precio, estoy encantado",
    "El mejor producto que he comprado en mucho tiempo",
    "Funciona perfecto, llegó antes de lo esperado",
    "Me encantó, muy fácil de usar y bonito diseño",
    "Gran compra, todos en casa están felices",
    "Producto de primera calidad, totalmente satisfecho",
    "Fantástico, cumple con todo lo prometido",
    "Muy contento con la compra, lo volvería a comprar",
    # Negativas
    "Pésima calidad, se rompió al segundo día",
    "No funciona como dice la descripción, decepcionado",
    "Muy malo, no lo recomiendo para nada",
    "El producto llegó dañado y el servicio es terrible",
    "Una pérdida de dinero, no vale lo que cuesta",
    "Horrible experiencia de compra, nunca más",
    "El peor producto que he probado, no sirve",
    "Mala calidad, el material es muy frágil",
    "No cumple con las expectativas, muy decepcionante",
    "Terrible, tuve que devolverlo inmediatamente"
]

etiquetas = ["positivo"] * 10 + ["negativo"] * 10

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    textos, etiquetas, test_size=0.3, random_state=42, stratify=etiquetas
)

print(f"Entrenamiento: {len(X_train)} documentos")
print(f"Prueba: {len(X_test)} documentos")

## 4.2 Naive Bayes

El clasificador **Naive Bayes** aplica el Teorema de Bayes con la suposición de independencia entre características:

$$P(c|d) = \frac{P(d|c) \cdot P(c)}{P(d)}$$

La variante **Multinomial Naive Bayes** es especialmente adecuada para texto, ya que modela las frecuencias de los términos.

### Ventajas
- Rápido de entrenar y predecir.
- Funciona bien con pocas muestras.
- Base sólida (*baseline*) para clasificación de texto.

In [ ]:
# Pipeline: TF-IDF + Naive Bayes
pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

pipe_nb.fit(X_train, y_train)
y_pred_nb = pipe_nb.predict(X_test)

print("=== Naive Bayes ===")
print(classification_report(y_test, y_pred_nb))

## 4.3 Support Vector Machine (SVM)

Las **SVM** buscan el hiperplano que maximiza el margen entre las clases. Son muy efectivas en espacios de alta dimensión como los generados por TF-IDF.

### ¿Por qué SVM funciona bien con texto?

- El texto genera vectores de alta dimensionalidad (muchos términos).
- Generalmente los datos son linealmente separables en espacios de alta dimensión.
- SVM maneja bien datos dispersos (*sparse*).

In [ ]:
# Pipeline: TF-IDF + SVM
pipe_svm = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(random_state=42, max_iter=10000))
])

pipe_svm.fit(X_train, y_train)
y_pred_svm = pipe_svm.predict(X_test)

print("=== SVM (Linear) ===")
print(classification_report(y_test, y_pred_svm))

## 4.4 Redes Neuronales (MLP)

Un **Perceptrón Multicapa** (MLP) puede capturar relaciones no lineales entre las características. Aunque para texto los modelos lineales suelen ser competitivos, las redes neuronales ofrecen mayor flexibilidad.

In [ ]:
# Pipeline: TF-IDF + MLP
pipe_mlp = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
])

pipe_mlp.fit(X_train, y_train)
y_pred_mlp = pipe_mlp.predict(X_test)

print("=== Red Neuronal (MLP) ===")
print(classification_report(y_test, y_pred_mlp))

## 4.5 Métricas de evaluación

| Métrica | Fórmula | Descripción |
|---------|---------|-------------|
| **Accuracy** | $\frac{TP + TN}{Total}$ | Proporción de predicciones correctas |
| **Precision** | $\frac{TP}{TP + FP}$ | De los predichos positivos, cuántos lo son realmente |
| **Recall** | $\frac{TP}{TP + FN}$ | De los realmente positivos, cuántos fueron detectados |
| **F1-Score** | $2 \cdot \frac{P \cdot R}{P + R}$ | Media armónica de Precision y Recall |

In [ ]:
# Matriz de confusión
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

modelos = [
    ('Naive Bayes', y_pred_nb),
    ('SVM', y_pred_svm),
    ('MLP', y_pred_mlp)
]

for ax, (nombre, y_pred) in zip(axes, modelos):
    cm = confusion_matrix(y_test, y_pred, labels=['positivo', 'negativo'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['positivo', 'negativo'])
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title(nombre)

plt.tight_layout()
plt.show()

## 4.6 Validación cruzada

La **validación cruzada** (*cross-validation*) es una técnica para evaluar modelos de forma más robusta, dividiendo los datos en $k$ pliegues (*folds*) y entrenando/evaluando $k$ veces.

In [ ]:
# Validación cruzada (5-fold)
print("Validación cruzada (5-fold):\n")

pipelines = {
    'Naive Bayes': Pipeline([('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())]),
    'SVM': Pipeline([('tfidf', TfidfVectorizer()), ('clf', LinearSVC(random_state=42, max_iter=10000))]),
    'MLP': Pipeline([('tfidf', TfidfVectorizer()), ('clf', MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42))])
}

for nombre, pipe in pipelines.items():
    scores = cross_val_score(pipe, textos, etiquetas, cv=5, scoring='f1_macro')
    print(f"  {nombre}: F1 = {scores.mean():.3f} (+/- {scores.std():.3f})")

## 4.7 Predicción de nuevos documentos

In [ ]:
# Predecir sentimiento de nuevas reseñas
nuevas_resenas = [
    "Me gustó mucho el producto, muy buena calidad",
    "No sirve para nada, estoy muy enojado",
    "Llegó rápido y bien empaquetado, excelente servicio",
    "Se descompuso a la semana, pésimo producto"
]

# Usar el mejor modelo (SVM)
predicciones = pipe_svm.predict(nuevas_resenas)

print("Predicciones para nuevas reseñas:\n")
for resena, pred in zip(nuevas_resenas, predicciones):
    emoji = "+" if pred == "positivo" else "-"
    print(f"  [{emoji}] {pred.upper()}: {resena}")

## Resumen

En este capítulo aprendimos:

- **Clasificación supervisada**: Requiere datos etiquetados para entrenar modelos predictivos.
- **Naive Bayes**: Rápido y efectivo como baseline para clasificación de texto.
- **SVM**: Excelente rendimiento en espacios de alta dimensión.
- **Redes Neuronales (MLP)**: Capturan relaciones no lineales.
- **Métricas**: Accuracy, Precision, Recall y F1-Score.
- **Validación cruzada**: Evaluación robusta del rendimiento del modelo.

En el próximo capítulo exploraremos los **Word Embeddings**, que ofrecen representaciones más ricas y densas del texto.